https://colab.research.google.com/github/cs221m/cs221m-course/blob/main/04_probes.ipynb

In [3]:
from IPython.display import clear_output
from nnsight import LanguageModel

model = LanguageModel('HuggingFaceTB/SmolLM2-1.7B')

with model.generate("Once upon a", max_new_tokens=1):
    outputs = model.generator.output.save()
    
model.to('cuda')

clear_output()

print(model.tokenizer.decode(outputs[0]))

Once upon a time


In [6]:
with model.generate("ES: amor, FR:"):
    last_layer_last_token = model.model.layers[-1].output[0, -1].save()
    output = model.generator.output.save()
    
clear_output()

print(model.tokenizer.decode(output))

['ES: amor, FR: amour, G: Liebe, IT: amore, IT: amore, RU']


In [11]:
print('Last layer shape: ', last_layer_last_token.shape)
print('Last layer first 10', last_layer_last_token[:10].tolist())

Last layer shape:  torch.Size([2048])
Last layer first 10 [77.5, -15.0625, -4.25, 45.75, -33.0, -57.5, -49.25, 3.390625, -19.375, 9.125]


In [16]:
unembed_layer = model.lm_head

logits = unembed_layer(model.model.norm(last_layer_last_token))

next_token_pred = logits.argmax(dim=-1)

next_token = model.tokenizer.decode(next_token_pred)

print('Next Token:', next_token)

Next Token:  am


In [25]:
import plotly.io as pio

pio.renderers.default = 'plotly_mimetype+notebook'

In [28]:
import torch
import plotly.express as px

prompt = "ES: amor, FR:"

layer_range = list(range(len(model.model.layers)))

decoded_layers = []
with model.trace(prompt):
    for layer_idx in layer_range:
        layer_act = model.model.layers[layer_idx].output
        layer_act = model.model.norm(layer_act)
        
        layer_logits = unembed_layer(layer_act)
        
        decoded_layer = torch.nn.functional.softmax(layer_logits, dim=-1)
        decoded_layers.append(decoded_layer.save())
        
decoded_layers = torch.cat(decoded_layers)

probabilities, decoded_tokens = decoded_layers.max(dim=-1)

decoded_tokens = [
    [model.tokenizer.decode(t.item()) for t in layer_tokens]
    for layer_tokens in decoded_tokens
]

input_tokens = [t.replace('Ġ', '') for t in model.tokenizer.tokenize(prompt)]

fig = px.imshow(
    probabilities.detach().cpu().float().numpy(),
    x=[f"{i+1}) {t}" for i, t in enumerate(input_tokens)],
    y=layer_range,
    origin="lower", # make 0 at the bottom
    color_continuous_scale=px.colors.diverging.RdYlBu,
    color_continuous_midpoint=0.50,
    text_auto=True,
    labels=dict(x="Input Tokens", y="Layers", color="Probability")
)

# prettify
fig.update_layout(
    title='Logit Lens for amor (ES) -> amour (FR)',
    xaxis_tickangle=0,
    width=1200,
    height=800
)

fig.update_xaxes(tickfont_size=18) 
fig.update_yaxes(tickvals=layer_range)
fig.update_traces(text=decoded_tokens, texttemplate="%{text}")

fig.show()

pca done decently alr, will be skipping for now